# DICE — Notebook 1.3 (Uncertainty, Monte Carlo) — Solutions

## 0) Setup and import

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["figure.dpi"] = 120
RNG = np.random.default_rng(42)

def clone_params(base, **overrides):
    candidate = Params()
    for name, value in vars(base).items():
        setattr(candidate, name, value)
    for name, value in overrides.items():
        setattr(candidate, name, float(value))
    return candidate

def simulate(base, **overrides):
    candidate = clone_params(base, **overrides)
    path = init_states(candidate)
    path[1:, candidate.i_s] = 0.20
    path[1:, candidate.i_mu] = np.linspace(0.03, 0.60, candidate.nT - 1)
    path = update_path(path, range(1, candidate.nT), candidate)
    return path, candidate

def damage_fraction(path, par):
    lagged_temperature = np.r_[path[0, par.i_T_AT], path[:-1, par.i_T_AT]]
    damages = par.a2 * lagged_temperature ** par.a3
    if par.a4 != 0:
        damages += np.where(lagged_temperature > par.a6,
                            par.a4 * lagged_temperature ** par.a5, 0.0)
    return damages

def quantile_bands(array):
    return np.quantile(np.asarray(array), [0.05, 0.50, 0.95], axis=0)


## Q1) Simulate the baseline simulation (one scenario).

Let us consider a baseline scenario. Simulate the DICE model with baseline calibration considering for controls (exogenous): constant saving `s_t = 0.20`; abatement `mu_t` ramps 0.03 -> 0.60 in 2060. Report path (`T_AT`) and  (`Y`) .

In [ ]:
p = Params()
baseline, _ = simulate(p)
years = baseline[:, p.i_time]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(years, baseline[:, p.i_T_AT], lw=2)
axes[0].set_title("Atmospheric temperature"); axes[0].set_ylabel("°C")
axes[1].plot(years, baseline[:, p.i_Y], lw=2)
axes[1].set_title("Gross output"); axes[1].set_ylabel("model units")
for ax in axes: ax.set_xlabel("Year"); ax.grid(True)
fig.tight_layout(); plt.show()


## Q2) Consider now parametric uncertainty about temperature sensitivity T2XCO2.

In Nordhaus (2018), one can read that:  
> *“Equilibrium temperature sensitivity (ETS). The distribution for ETS adopts the approach used in the MUP study. The primary estimates are from Olsen et al. (2012). This study uses a Bayesian approach, with a prior based on previous studies and a likelihood based on observational or modeled data. The best fitting distribution is a log-normal PDF. The parameters of the log-normal distribution fit to Olsen et al. are μ = 1.107 and σ = 0.264. The major summary statistics of the reference distribution in the study are the following: mean = 3.13, median = 3.03 and standard deviation = 0.843.”*


### Q2A) Draw 1000 observations of T2XCO2 and report the histogram.
*Hints:*
- Use `np.random.lognormal(mean=1.107, sigma=0.264, size=1000)` to draw samples.  
- Plot the histogram with `plt.hist(samples, bins=30, density=True)`.  
- Check summary statistics with `np.mean`, `np.median`, `np.std`, `np.quantile(samples,[0.05,0.5,0.95])`.


In [ ]:
N = 1000
# Lognormal distribution centred on the default equilibrium climate sensitivity.
sigma_ecs = math.log(1.25) / norm.ppf(0.95)
ecs_draws = np.exp(math.log(p.T2XCO2) + sigma_ecs * RNG.standard_normal(N))
plt.hist(ecs_draws, bins=35, edgecolor="white")
plt.xlabel("Equilibrium climate sensitivity T2XCO2 (°C)")
plt.ylabel("Count"); plt.title("Q2A — ECS draws"); plt.show()
print("ECS 5/50/95%:", np.round(np.quantile(ecs_draws, [0.05, 0.5, 0.95]), 3))


### Q2B) Create a loop that generates the update path and stores them.
*Hints:*
- Start with `p = Params()`, `sim = init_states(p)`.  
- For each draw, create a new parameter object and override: `p2.T2XCO2 = float(t2x)`.  
- Set controls as in baseline (constant `s`, linear ramp for `mu`).  
- Run `update_path(sim2, range(1, p2.nT), p2)`.  
- Store results in lists, e.g. `T_list.append(sim2[:, p2.i_T_AT])`, `Y_list.append(sim2[:, p2.i_Y])`, and damages `D_list`  (compute it manually).


In [ ]:
T_ecs, Y_ecs, D_ecs = [], [], []
for ecs in ecs_draws:
    path, par = simulate(p, T2XCO2=ecs)
    T_ecs.append(path[:, par.i_T_AT])
    Y_ecs.append(path[:, par.i_Y])
    D_ecs.append(damage_fraction(path, par))
T_ecs, Y_ecs, D_ecs = map(np.asarray, (T_ecs, Y_ecs, D_ecs))
print("Stored arrays:", T_ecs.shape, Y_ecs.shape, D_ecs.shape)



### Q2C) Report the confidence interval for temperatures, GDP and damages.
*Hints:*
- Use `np.quantile(arr, [0.05,0.5,0.95], axis=0)` to compute 5–50–95% bands.  
- Plot fan charts with `plt.fill_between(years, q05, q95, alpha=0.2)` and `plt.plot(years, q50)`.  
- Extract values in 2100 with `i2100 = np.argmin(np.abs(years-2100))` and print `q05[i2100], q50[i2100], q95[i2100]`.  


In [ ]:
bands_ecs = [quantile_bands(x) for x in (T_ecs, Y_ecs, D_ecs)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, bands, title, unit in zip(axes, bands_ecs,
        ["Temperature", "Gross output", "Damage fraction"], ["°C", "model units", "share"]):
    q05, q50, q95 = bands
    ax.fill_between(years, q05, q95, alpha=.25); ax.plot(years, q50, lw=2)
    ax.set_title(title); ax.set_xlabel("Year"); ax.set_ylabel(unit); ax.grid(True)
fig.tight_layout(); plt.show()
idx_2100 = int(np.argmin(np.abs(years - 2100)))
print("2100 ECS-only temperature 5/50/95%:",
      np.round(bands_ecs[0][:, idx_2100], 3), "°C")


> **Interpretation.** These are conditional Monte Carlo bands generated from an assumed parameter distribution. They are neither forecast intervals nor probabilities attached to SSP/NGFS scenarios. The distributional choice and model structure remain sources of deep uncertainty.

## Q3) Consider now parametric uncertainty about the damage parameter a2.

In Nordhaus (2018), one can read that:  
> *“Given the different approaches, I settled on a value for the uncertainty of the damage parameter which is one-half the mean value of the parameter. More precisely, the distribution is assumed to be normal, with a standard deviation of 0.118% Y/°C². This reflects the great divergence today among different studies.”*



### Q3A) Draw 1000 observations of the damage parameter a2 and report the histogram.
*Hints:*
- Use `np.random.normal(loc=baseline, scale=0.00118, size=1000)` where `baseline = p.a2`.  
- If you want to prevent negative values, apply a truncation (e.g. discard draws `< 0`).  
- Plot the histogram with `plt.hist(samples, bins=30, density=True)`.  
- Check summary statistics with `np.mean`, `np.median`, `np.std`, and `np.quantile`.

In [ ]:
sigma_a2 = math.log(2.0) / norm.ppf(0.95)
a2_draws = np.exp(math.log(p.a2) + sigma_a2 * RNG.standard_normal(N))
plt.hist(a2_draws, bins=35, edgecolor="white")
plt.xlabel("Quadratic damage coefficient a2"); plt.ylabel("Count")
plt.title("Q3A — Damage-parameter draws"); plt.show()
print("a2 5/50/95%:", np.round(np.quantile(a2_draws, [0.05, 0.5, 0.95]), 6))


### Q3B) Create a loop that generates the update path and stores them.
*Hints:*
- Start with `p = Params()` and `sim = init_states(p)`.  
- For each draw, override `p2.a2 = float(a2draw)`.  
- Set exogenous controls (`s`, `mu`) as in baseline.  
- Run `update_path(sim2, range(1, p2.nT), p2)` and store results for `T_AT`, `Y`, and damages.  

In [ ]:
T_ecs_a2, Y_ecs_a2, D_ecs_a2 = [], [], []
for ecs, a2 in zip(ecs_draws, a2_draws):
    path, par = simulate(p, T2XCO2=ecs, a2=a2)
    T_ecs_a2.append(path[:, par.i_T_AT])
    Y_ecs_a2.append(path[:, par.i_Y])
    D_ecs_a2.append(damage_fraction(path, par))
T_ecs_a2, Y_ecs_a2, D_ecs_a2 = map(np.asarray, (T_ecs_a2, Y_ecs_a2, D_ecs_a2))


### Q3C) Report the confidence interval for temperatures, GDP and damages.  Interpret.
*Hints:*
- Use `np.quantile(arr, [0.05,0.5,0.95], axis=0)` across the simulated ensemble.  
- Plot fan charts with `plt.fill_between(years, q05, q95, alpha=0.2)` and `plt.plot(years, q50)`.  
- Report values for 2100: find index with `i2100 = np.argmin(np.abs(years-2100))` and print the quantiles for `T_AT`, `Y`, and damages.  

In [ ]:
bands_ecs_a2 = [quantile_bands(x) for x in (T_ecs_a2, Y_ecs_a2, D_ecs_a2)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, bands, title, unit in zip(axes, bands_ecs_a2,
        ["Temperature", "Gross output", "Damage fraction"], ["°C", "model units", "share"]):
    q05, q50, q95 = bands
    ax.fill_between(years, q05, q95, alpha=.25); ax.plot(years, q50, lw=2)
    ax.set_title(title); ax.set_xlabel("Year"); ax.set_ylabel(unit); ax.grid(True)
fig.tight_layout(); plt.show()


> **Interpretation.** Adding damage-function uncertainty widens the distribution of damages and economic outputs. It need not widen temperature bands much because the imposed policy path is not re-optimised in response to damages.

 ### Q3D) Compare how the additional source of uncertainty increase the size of the confidence intervals. Plot the two confidence intervals for temperatures.

In [ ]:
# Plot the two temperature bands and compare their 2100 widths.
fig, ax = plt.subplots(figsize=(8, 4))
q05, q50, q95 = bands_ecs[0]
ax.fill_between(years, q05, q95, alpha=0.20, label="ECS only: 5–95%")
ax.plot(years, q50, lw=2)
q05j, q50j, q95j = bands_ecs_a2[0]
ax.fill_between(years, q05j, q95j, alpha=0.20, label="ECS + a2: 5–95%")
ax.plot(years, q50j, lw=2)
ax.set_title("Temperature uncertainty bands")
ax.set_xlabel("Year"); ax.set_ylabel("°C"); ax.grid(True); ax.legend()
fig.tight_layout(); plt.show()

labels = ["Temperature", "Gross output", "Damage fraction"]
for label, ecs_band, joint_band in zip(labels, bands_ecs, bands_ecs_a2):
    width_ecs = ecs_band[2, idx_2100] - ecs_band[0, idx_2100]
    width_joint = joint_band[2, idx_2100] - joint_band[0, idx_2100]
    print(f"{label}: 2100 5–95 width ECS-only={width_ecs:.4g}; ECS+a2={width_joint:.4g}")


> **Interpretation.** Compare widths variable by variable. Parameter uncertainty propagates through specific model channels; it must not be collapsed into one confidence score or treated as covering structural and scenario uncertainty.

## Q4) Consider now parametric uncertainty about the decarbonization parameter σ(t).

In Nordhaus (2018), one can read that:  
> *“The simplest approach is to estimate an OLS regression, using data from 1960 to 2015, and then look at the forecast error for 2100. If an AR1 term is included in the equation, the standard error of the forecast for 2100 is 13.5% of the logarithm of σ(t). This implies an annual uncertainty of 0.149% per year. However, a unit root of σ(t) cannot be rejected, so this estimate is biased downward.”*


### Q4A) Draw 1000 observations of the decarbonization trend parameter.
*Hints:*
- Use a **normal distribution** centered at the baseline value (`p.deltasig`),  
  with `scale = 0.00149` (≈ 0.149% per year).  
- To ensure strictly positive values, either filter out negatives or use a truncated normal (`draw_trunc_normal`).  
- Example: `np.random.normal(loc=p.deltasig, scale=0.00149, size=1000)`.


In [ ]:
# The baseline value is zero, so use a transparent half-normal stress distribution.
deltasig_draws = np.abs(RNG.normal(loc=0.0, scale=0.02, size=N))
plt.hist(deltasig_draws, bins=35, edgecolor="white")
plt.xlabel("deltasig"); plt.ylabel("Count")
plt.title("Q4A — Decarbonisation-trend draws"); plt.show()
print("deltasig 5/50/95%:", np.round(np.quantile(deltasig_draws, [0.05, 0.5, 0.95]), 4))



### Q4B) Create a loop that generates the update path and stores them.
*Hints:*
- Start with `p = Params()`, `sim = init_states(p)`.  
- For each draw, override `p2.deltasig = float(draw)`.  
- Keep controls (`s`, `mu`) fixed as in the baseline.  
- Run `update_path(sim2, range(1, p2.nT), p2)` and store results for `T_AT`, `Y`, and damages.


In [ ]:
T_all, Y_all, D_all = [], [], []
for ecs, a2, deltasig in zip(ecs_draws, a2_draws, deltasig_draws):
    path, par = simulate(p, T2XCO2=ecs, a2=a2, deltasig=deltasig)
    T_all.append(path[:, par.i_T_AT])
    Y_all.append(path[:, par.i_Y])
    D_all.append(damage_fraction(path, par))
T_all, Y_all, D_all = map(np.asarray, (T_all, Y_all, D_all))


### Q4C) Report the confidence interval for temperatures, GDP and damages.
*Hints:*
- Use `np.quantile(arr, [0.05,0.5,0.95], axis=0)` to compute confidence bands across the ensemble.  
- Plot fan charts with `plt.fill_between(years, q05, q95, alpha=0.2)` and overlay the median with `plt.plot(years, q50)`.  
- Extract 2100 values with `i2100 = np.argmin(np.abs(years-2100))` and print the 5–50–95% range.  

In [ ]:
bands_all = [quantile_bands(x) for x in (T_all, Y_all, D_all)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, bands, title, unit in zip(axes, bands_all,
        ["Temperature", "Gross output", "Damage fraction"], ["°C", "model units", "share"]):
    q05, q50, q95 = bands
    ax.fill_between(years, q05, q95, alpha=.25); ax.plot(years, q50, lw=2)
    ax.set_title(title); ax.set_xlabel("Year"); ax.set_ylabel(unit); ax.grid(True)
fig.tight_layout(); plt.show()
for label, joint_band, all_band in zip(["Temperature", "Gross output", "Damage fraction"], bands_ecs_a2, bands_all):
    width_joint = joint_band[2, idx_2100] - joint_band[0, idx_2100]
    width_all = all_band[2, idx_2100] - all_band[0, idx_2100]
    print(f"{label}: 2100 5–95 width ECS+a2={width_joint:.4g}; +deltasig={width_all:.4g}")


In [ ]:
# Intentionally left as a workspace cell.